<a href="https://colab.research.google.com/github/ashwiniv07/A-Synergistic-Deep-Fusion-and-Adversarial-Defense-Framework-/blob/main/STAMP_1_Data_Preparation_FINAL(D).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===================== 1. Setup & configuration =====================
import os, re, gc, json, time, csv
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from google.colab import drive
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/Paper4_STAMP')
RAW, PROC = BASE / 'raw', BASE / 'processed'
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)

DATASET = 'partB'      # 'partB' (billing) or 'partD' (prescriptions)
MODE = 'prospective'   # 'main' = 2017-2022 study | 'prospective' = 2022-2024 check (use with partB)    # 'main' = 2017-2022 study | 'prospective' = 2022-2024 check (use with partB)

if MODE == 'main':
    YEARS, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = [2017, 2018, 2019, 2020, 2021, 2022], [2017, 2018, 2019], [2020], [2021, 2022]
    OUT_TAG = DATASET
else:                  # 2022 only provides last-year values for the drift features of 2023
    YEARS, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = [2022, 2023, 2024], [2022], [], [2023, 2024]
    OUT_TAG = f'{DATASET}_prosp'
EXPOSURE_COL = {'partB': 'Tot_Mdcr_Pymt_Amt', 'partD': 'Tot_Drug_Cst'}[DATASET]

LOOKBACK_YEARS = 5
STRICT_TYPES = {'1128a1', '1128a3', '1128b1', '1128b7'}
BROAD_TYPES = STRICT_TYPES | {'1128a2', '1128b4', '1128c3gi', '1128c3gii'}
LABEL_SET = 'strict'
FRAUD_EXCL_TYPES = STRICT_TYPES if LABEL_SET == 'strict' else BROAD_TYPES
DROP_OTHER_EXCLUDED = True
SUPPLEMENT_MONTHS_BACK = 30
MIN_PEERS, SEED = 30, 42
np.random.seed(SEED)
print(f'{DATASET} | {MODE} | years {YEARS} | outputs prefixed {OUT_TAG}_ | exposure = {EXPOSURE_COL}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
partB | prospective | years [2022, 2023, 2024] | outputs prefixed partB_prosp_ | exposure = Tot_Mdcr_Pymt_Amt


In [ ]:
# ===================== 2. Download CMS yearly files (skips files already in Drive) + LEIE =====================
HEADERS = {'User-Agent': 'Mozilla/5.0 (research data download; Colab)'}
TITLES = {'partB': 'medicare physician & other practitioners - by provider',
          'partD': 'medicare part d prescribers - by provider'}
def raw_file(year):
    return RAW / f'{DATASET}_{year}.csv'

def download(url, dest, chunk=1 << 20):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return 'already have'
    tmp = dest.with_suffix(dest.suffix + '.part')
    with requests.get(url, headers=HEADERS, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(tmp, 'wb') as f:
            for block in r.iter_content(chunk):
                f.write(block)
    tmp.rename(dest)
    return f'downloaded ({dest.stat().st_size/1e6:.0f} MB)'

missing = [y for y in YEARS if not raw_file(y).exists()]
if missing:
    cat = requests.get('https://data.cms.gov/data.json', headers=HEADERS, timeout=180).json()
    CSV_URLS = {}
    for d in cat['dataset']:
        for dist in [d] + d.get('distribution', []):
            t = str(dist.get('title', ''))
            if t.split(' : ')[0].strip().lower() != TITLES[DATASET]:
                continue
            m = re.search(r' : (20\d\d)', t)
            fmt = str(dist.get('format') or dist.get('mediaType') or '').lower()
            if m and 'csv' in fmt and dist.get('downloadURL'):
                CSV_URLS.setdefault(int(m.group(1)), dist['downloadURL'])
    print('CSV links found for years:', sorted(CSV_URLS))
    for y in missing:
        try:
            print(y, download(CSV_URLS[y], raw_file(y)) if y in CSV_URLS else 'no link')
        except Exception as e:
            print(y, 'FAILED:', e)

LEIE_FILE = RAW / 'LEIE_UPDATED.csv'
SUPP_DIR = RAW / 'leie_supplements'; SUPP_DIR.mkdir(exist_ok=True)
EXTRA_DIR = RAW / 'leie_extra'; EXTRA_DIR.mkdir(exist_ok=True)
if not LEIE_FILE.exists():
    print('LEIE:', download('https://oig.hhs.gov/exclusions/downloadables/UPDATED.csv', LEIE_FILE))
print('MISSING CMS years:', [y for y in YEARS if not raw_file(y).exists()] or 'none')
print('LEIE present:', LEIE_FILE.exists(), '| reinstatement files:', len(list(SUPP_DIR.glob('*.csv'))))

MISSING CMS years: none
LEIE present: True | reinstatement files: 22


In [ ]:
# ===== 2b. Download any missing CMS years again (with retries), then check =====
import time, re, requests
miss = [y for y in YEARS if not raw_file(y).exists() or raw_file(y).stat().st_size == 0]
print('Missing before:', miss or 'none')
if miss:
    cat = requests.get('https://data.cms.gov/data.json', headers=HEADERS, timeout=180).json()
    urls = {}
    for d in cat['dataset']:
        for dist in [d] + d.get('distribution', []):
            t = str(dist.get('title', ''))
            if t.split(' : ')[0].strip().lower() != TITLES[DATASET]:
                continue
            m = re.search(r' : (20\d\d)', t)
            url = dist.get('downloadURL') or dist.get('accessURL')
            fmt = str(dist.get('format') or dist.get('mediaType') or '').lower()
            if m and url and ('csv' in fmt or str(url).lower().split('?')[0].endswith('.csv')):
                urls.setdefault(int(m.group(1)), url)
    for y in miss:
        if y not in urls:
            print(y, 'NO LINK in catalog -> send screenshot'); continue
        for attempt in range(1, 4):
            try:
                tmp = raw_file(y).with_suffix('.csv.part')
                with requests.get(urls[y], headers=HEADERS, stream=True, timeout=900) as r:
                    r.raise_for_status()
                    with open(tmp, 'wb') as f:
                        for block in r.iter_content(1 << 20):
                            f.write(block)
                tmp.rename(raw_file(y))
                print(y, f'downloaded ({raw_file(y).stat().st_size/1e6:.0f} MB)'); break
            except Exception as e:
                print(y, f'attempt {attempt} failed: {e}'); time.sleep(10)

still = [y for y in YEARS if not raw_file(y).exists()]
print('MISSING CMS years:', still or 'none  -> OK, continue to cell 3')
assert not still, 'Some years still missing: run this cell again'

Missing before: none
MISSING CMS years: none  -> OK, continue to cell 3


In [ ]:
# ===================== 3. Read each year (full population): keep NPI, specialty, state + numeric columns =====================
# Personal/address text columns are dropped. Codes that only look numeric (ZIP, FIPS) are dropped too.
DROP_PAT = re.compile(r'(last_org_name|first_name|_mi$|crdntls|_st1$|_st2$|city|zip|fips|cntry|ruca_desc|gndr)', re.I)

def find_col(cols, pattern):
    hits = [c for c in cols if re.search(pattern, c, re.I)]
    return hits[0] if hits else None

def process_year(year, chunksize=250_000):
    path = raw_file(year)
    header = pd.read_csv(path, nrows=0).columns.tolist()
    npi_c   = find_col(header, r'npi$')
    type_c  = find_col(header, r'(prvdr_type|prscrbr_type)$')
    state_c = find_col(header, r'state_abrvtn$')
    ent_c   = find_col(header, r'ent_cd$')                 # I = individual, O = organisation (Part B only)
    if npi_c is None or type_c is None or state_c is None:
        raise ValueError(f'{year}: could not find NPI / type / state columns in {header[:15]}')
    cat_cols = [c for c in (npi_c, type_c, state_c, ent_c) if c]
    supp_cols = [c for c in header if re.search(r'sprsn', c, re.I)]          # suppression flags
    other = [c for c in header if c not in cat_cols + supp_cols and not DROP_PAT.search(c)]

    parts, numeric_cols = [], None
    for chunk in pd.read_csv(path, dtype=str, chunksize=chunksize, low_memory=False):
        if numeric_cols is None:     # decide which columns are numeric from the first chunk
            numeric_cols = []
            for c in other:
                s = chunk[c].dropna()
                if len(s) == 0 or pd.to_numeric(s.str.replace(',', ''), errors='coerce').notna().mean() > 0.9:
                    numeric_cols.append(c)
        out = pd.DataFrame({
            'npi':      chunk[npi_c].str.strip(),
            'prov_type': chunk[type_c].fillna('Unknown').str.strip(),
            'state':    chunk[state_c].fillna('NA').str.strip(),
            'entity':   chunk[ent_c].fillna('NA').str.strip() if ent_c else 'NA',
        })
        for c in numeric_cols:
            out[c] = pd.to_numeric(chunk[c].str.replace(',', ''), errors='coerce').astype('float32')
        for c in supp_cols:
            out[c + '_flag'] = chunk[c].notna().astype('int8')     # 1 = value was suppressed by CMS
        parts.append(out)
    df = pd.concat(parts, ignore_index=True)
    df['year'] = year
    df = df[df['npi'].str.fullmatch(r'\d{10}', na=False)]
    print(f'{year}: {len(df):,} providers, {len(numeric_cols)} numeric columns')
    return df

frames = []
for y in YEARS:
    if not raw_file(y).exists():
        raise FileNotFoundError(f'{y} missing - run cell 2 again')
    f = process_year(y)
    for c in ('prov_type', 'state', 'entity'):          # text -> category saves ~1.5 GB
        f[c] = f[c].astype('category')
    frames.append(f)
    del f; gc.collect()

# keep only columns present in every year, give all years the same category lists
common = set.intersection(*[set(f.columns) for f in frames])
ordered = [c for c in frames[0].columns if c in common]
for c in ('prov_type', 'state', 'entity'):
    cats = sorted(set().union(*[set(f[c].cat.categories) for f in frames]))
    for f in frames:
        f[c] = f[c].cat.set_categories(cats)
for i in range(len(frames)):
    frames[i] = frames[i][ordered]
gc.collect()
data = pd.concat(frames, ignore_index=True)
del frames; gc.collect()
assert (data.groupby('npi', sort=False)['year'].diff().dropna() > 0).all(), 'rows not in year order'
print('Combined:', data.shape, '| memory GB:', round(data.memory_usage(deep=True).sum() / 1e9, 2))

2022: 1,230,293 providers, 63 numeric columns
2023: 1,259,343 providers, 63 numeric columns
2024: 1,296,739 providers, 63 numeric columns
Combined: (3786375, 70) | memory GB: 1.23


In [ ]:
# ===================== 4. Ratio features + peer-deviation features (full population, no labels) =====================
# Ratios describe HOW a provider bills; peer deviation says how unusual that is versus similar doctors.
def safe_div(a, b):
    return (a / b.replace(0, np.nan)).astype('float32')

C = data.columns
def col(name):
    return name if name in C else None

if DATASET == 'partB':
    ratio_specs = {
        'srv_per_bene':       ('Tot_Srvcs', 'Tot_Benes'),               # phantom billing / overuse
        'chrg_per_srv':       ('Tot_Sbmtd_Chrg', 'Tot_Srvcs'),          # upcoding
        'pay_per_bene':       ('Tot_Mdcr_Pymt_Amt', 'Tot_Benes'),
        'sbmt_to_allowed':    ('Tot_Sbmtd_Chrg', 'Tot_Mdcr_Alowd_Amt'), # charges far above allowed
        'codes_per_bene':     ('Tot_HCPCS_Cds', 'Tot_Benes'),           # unbundling
        'srv_per_code':       ('Tot_Srvcs', 'Tot_HCPCS_Cds'),
        'drug_share_pay':     ('Drug_Mdcr_Pymt_Amt', 'Tot_Mdcr_Pymt_Amt'),
    }
else:  # partD
    ratio_specs = {
        'clms_per_bene':      ('Tot_Clms', 'Tot_Benes'),
        'cost_per_clm':       ('Tot_Drug_Cst', 'Tot_Clms'),
        'days_per_clm':       ('Tot_Day_Suply', 'Tot_Clms'),
        'opioid_share':       ('Opioid_Tot_Clms', 'Tot_Clms'),
        'brand_share':        ('Brnd_Tot_Clms', 'Tot_Clms'),
        'cost_per_bene':      ('Tot_Drug_Cst', 'Tot_Benes'),
    }

ratio_cols = []
for name, (a, b) in ratio_specs.items():
    if col(a) and col(b):
        data[name] = safe_div(data[a], data[b])
        ratio_cols.append(name)
    else:
        print(f'skip {name}: column {a if not col(a) else b} not in this dataset')

totals = [c for c in ('Tot_Benes', 'Tot_Srvcs', 'Tot_Sbmtd_Chrg', 'Tot_Mdcr_Pymt_Amt',
                      'Tot_Clms', 'Tot_Drug_Cst') if col(c)]
peer_base = ratio_cols + totals

def robust_z(df, value, keys):
    """(x - peer median) / (1.4826 * peer MAD) on log1p scale, computed within each peer group."""
    x = np.log1p(df[value].clip(lower=0))
    g = x.groupby([df[k] for k in keys], observed=True)
    med = g.transform('median')
    mad = (x - med).abs().groupby([df[k] for k in keys], observed=True).transform('median')
    return ((x - med) / (1.4826 * mad + 1e-6)).astype('float32')

# peer group = same year + specialty + state; small groups fall back to year + specialty
grp_size = data.groupby(['year', 'prov_type', 'state'], observed=True)['npi'].transform('size')
small = grp_size < MIN_PEERS
for v in peer_base:
    z_fine   = robust_z(data, v, ['year', 'prov_type', 'state'])
    z_coarse = robust_z(data, v, ['year', 'prov_type'])
    data[f'peerz_{v}'] = z_fine.where(~small, z_coarse).clip(-20, 20)
    data[f'peerpct_{v}'] = data.groupby(['year', 'prov_type'], observed=True)[v].rank(pct=True).astype('float32')
data['peer_group_size'] = grp_size.astype('float32')
print('Peer features added:', 2 * len(peer_base))

Peer features added: 22


In [ ]:
# ===================== 5. Temporal drift features (full population, past years only, no labels) =====================
t0 = time.time()
g = data.groupby('npi', sort=False)
prev_year = g['year'].shift(1)
consec = (data['year'] - prev_year) == 1

base = ratio_cols + totals
peer = [c for c in data.columns if c.startswith('peerz_')]
DRIFT = []
for c in base:
    lag = g[c].shift(1).where(consec)
    data['d_' + c] = (np.log1p(data[c].clip(lower=0)) - np.log1p(lag.clip(lower=0))).astype('float32')
    DRIFT.append('d_' + c)
for c in peer:
    lag = g[c].shift(1).where(consec)
    data['dz_' + c[6:]] = (data[c] - lag).astype('float32')
    DRIFT.append('dz_' + c[6:])

first_year = data['year'].min()
data['is_new'] = np.where(data['year'] == first_year, np.nan, prev_year.isna().astype('float32')).astype('float32')
data['returned_after_gap'] = (prev_year.notna() & ~consec).astype('float32')
dz = data[[c for c in DRIFT if c.startswith('dz_')]]
data['max_abs_peer_shift'] = dz.abs().max(axis=1).astype('float32')
data['n_large_shifts'] = (dz.abs() > 2).sum(axis=1).astype('float32')
DRIFT += ['is_new', 'returned_after_gap', 'max_abs_peer_shift', 'n_large_shifts']
del dz, g, prev_year, consec, lag
gc.collect()
print(f'{len(DRIFT)} drift features in {time.time()-t0:.0f}s | memory GB:', round(data.memory_usage(deep=True).sum() / 1e9, 2))

26 drift features in 7s | memory GB: 2.07


In [ ]:
# ===================== 6. CHECKPOINT: save features per year, then restart =====================
CK = PROC / f'features_ckpt_{OUT_TAG}'
CK.mkdir(exist_ok=True)
for y in YEARS:
    data[data['year'] == y].to_parquet(CK / f'{y}.parquet', index=False)
    print('saved features for', y)
json.dump({'ratio_cols': ratio_cols, 'peer_base': peer_base, 'drift_cols': DRIFT},
          open(CK / 'ckpt_meta.json', 'w'), indent=2)
print('\nCheckpoint saved in', CK)
print('NOW: Runtime -> Restart session, then run ONLY cells 1, 2, 7a and 7b (skip 3-6).')

saved features for 2022
saved features for 2023
saved features for 2024

Checkpoint saved in /content/drive/MyDrive/Paper4_STAMP/processed/features_ckpt_partB_prosp
NOW: Runtime -> Restart session, then run ONLY cells 1, 2, 7a and 7b (skip 3-6).


In [ ]:
# ===================== 7a. LEIE + helpers (after restart: run cells 1, 2, then 7a, 7b) =====================
import pyarrow as pa, pyarrow.parquet as pq
CK = PROC / f'features_ckpt_{OUT_TAG}'
ck = json.load(open(CK / 'ckpt_meta.json'))
ratio_cols, peer_base, DRIFT = ck['ratio_cols'], ck['peer_base'], ck['drift_cols']

def read_leie(path, source):
    d = pd.read_csv(path, dtype=str, encoding_errors='replace')
    d.columns = d.columns.str.upper().str.strip()
    if not {'NPI', 'EXCLTYPE', 'EXCLDATE'}.issubset(d.columns):
        return None
    d['SOURCE'] = source
    return d[['NPI', 'EXCLTYPE', 'EXCLDATE', 'SOURCE']]

parts = [read_leie(LEIE_FILE, 'updated')] + [read_leie(p, 'reinstatement') for p in sorted(SUPP_DIR.glob('*.csv'))]
parts += [read_leie(p, 'snapshot') for p in sorted(EXTRA_DIR.glob('*.csv'))]
leie = pd.concat([p for p in parts if p is not None], ignore_index=True)
leie['NPI'] = leie['NPI'].fillna('').str.strip()
leie = leie[leie['NPI'].str.fullmatch(r'\d{10}') & (leie['NPI'] != '0000000000')].copy()
leie['TYPE_N'] = leie['EXCLTYPE'].fillna('').str.lower().str.replace(r'[^0-9a-z]', '', regex=True)
leie['EXCL_YEAR'] = pd.to_datetime(leie['EXCLDATE'], format='%Y%m%d', errors='coerce').dt.year
leie = leie.dropna(subset=['EXCL_YEAR']).drop_duplicates(subset=['NPI', 'TYPE_N', 'EXCLDATE'])
excl_strict = leie[leie['TYPE_N'].isin(STRICT_TYPES)].groupby('NPI')['EXCL_YEAR'].min()
excl_broad = leie[leie['TYPE_N'].isin(BROAD_TYPES)].groupby('NPI')['EXCL_YEAR'].min()
excl_chosen = excl_strict if LABEL_SET == 'strict' else excl_broad
other_excl = set(leie['NPI']) - set(excl_chosen.index)

def label_and_clean(df, is_test):
    gap_s = df['npi'].map(excl_strict) - df['year']
    gap_b = df['npi'].map(excl_broad) - df['year']
    gap = gap_s if LABEL_SET == 'strict' else gap_b
    df['label_strict'] = gap_s.between(0, LOOKBACK_YEARS).astype('int8')
    df['label_broad'] = gap_b.between(0, LOOKBACK_YEARS).astype('int8')
    df['label'] = df['label_strict'] if LABEL_SET == 'strict' else df['label_broad']
    df['excl_year'] = df['npi'].map(excl_chosen).astype('float32')
    drop = gap < 0
    if not is_test:
        drop = drop | (gap > LOOKBACK_YEARS)
        if DROP_OTHER_EXCLUDED:
            drop = drop | df['npi'].isin(other_excl)
    return df[~drop].reset_index(drop=True), int(drop.sum())

print('7a done | LEIE records:', len(leie))

7a done | LEIE records: 9288


In [ ]:
# ===================== 7b. Label + save, one YEAR at a time =====================
meta_cols = ['npi', 'year', 'split', 'label', 'label_strict', 'label_broad', 'excl_year', 'unseen_provider', 'exposure_usd']
cat_cols = ['prov_type', 'state', 'entity']
forbidden = re.compile(r'(label|excl|leie|rein|unseen|split|exposure)', re.I)
summary, train_npis, feature_cols = [], set(), None

for split_name, years in (('train', TRAIN_YEARS), ('val', VAL_YEARS), ('test', TEST_YEARS)):
    if not years:
        continue
    writer, path = None, PROC / f'{OUT_TAG}_{split_name}.parquet'
    for y in years:
        df = pd.read_parquet(CK / f'{y}.parquet')
        df, removed = label_and_clean(df, is_test=(split_name == 'test'))
        if split_name == 'train':
            train_npis |= set(df['npi'])
        df['split'] = split_name
        df['unseen_provider'] = (~df['npi'].isin(train_npis)).astype('int8')
        df['exposure_usd'] = df[EXPOSURE_COL].astype('float32')
        for c in cat_cols:
            df[c] = df[c].astype(str)
        if feature_cols is None:
            feature_cols = [c for c in df.columns if c not in meta_cols + cat_cols + DRIFT]
            assert not [c for c in feature_cols + DRIFT if forbidden.search(c)], 'leakage'
        table = pa.Table.from_pandas(df, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(path, table.schema)
        writer.write_table(table.cast(writer.schema))
        summary.append({'year': y, 'split': split_name, 'rows': len(df), 'fraud': int(df.label.sum()),
                        'removed': removed,
                        'is_new_fraud': round(float(df.loc[df.label == 1, 'is_new'].mean()), 4),
                        'is_new_nonfraud': round(float(df.loc[df.label == 0, 'is_new'].mean()), 4)})
        print(split_name, y, f'{len(df):,} rows | fraud {int(df.label.sum())}')
        del df, table
        gc.collect()
    writer.close()

meta = {'dataset': DATASET, 'mode': MODE, 'years': YEARS, 'train_years': TRAIN_YEARS, 'val_years': VAL_YEARS,
        'test_years': TEST_YEARS, 'lookback_years': LOOKBACK_YEARS, 'label_set': LABEL_SET,
        'strict_types': sorted(STRICT_TYPES), 'broad_types': sorted(BROAD_TYPES),
        'drop_other_excluded': DROP_OTHER_EXCLUDED, 'exposure_source_col': EXPOSURE_COL,
        'cat_cols': cat_cols, 'feature_cols': feature_cols, 'drift_cols': DRIFT,
        'ratio_cols': ratio_cols, 'peer_base': peer_base, 'meta_cols': meta_cols,
        'features_before_cleaning': True}
json.dump(meta, open(PROC / f'{OUT_TAG}_meta.json', 'w'), indent=2)
print(f'\nSaved {OUT_TAG}_*.parquet. Base features:', len(feature_cols), '| drift features:', len(DRIFT))
print(pd.DataFrame(summary).to_string(index=False))

train 2022 1,230,007 rows | fraud 182
test 2023 1,259,341 rows | fraud 110
test 2024 1,296,738 rows | fraud 61

Saved partB_prosp_*.parquet. Base features: 95 | drift features: 26
 year split    rows  fraud  removed  is_new_fraud  is_new_nonfraud
 2022 train 1230007    182      286           NaN              NaN
 2023  test 1259341    110        2        0.0636           0.0988
 2024  test 1296738     61        1        0.0328           0.0913
